# Evasion: Alternate Data Streams

| Field | Value |
|---|---|
| MITRE ATT&CK | T1564 |
| Tactic | Evasion |
| Difficulty | Intermediate |
| Time | ~30 min |

## Threat Context

In 2019, the Astaroth (Guildma) trojan targeted Brazilian banking customers using a sophisticated delivery chain that leveraged NTFS Alternate Data Streams (ADS) to hide its payload. After initial delivery via spearphishing, the malware stored its components inside ADS attached to legitimate files, making the malicious content invisible to standard directory listings and many security tools.

NTFS Alternate Data Streams are a feature of the Windows NTFS filesystem that allows multiple data streams to be associated with a single file. The main stream (the one you see when you open a file) is the default, but additional streams can store arbitrary data — and they don't show up in `dir` output, Windows Explorer, or most file managers.

Astaroth used ADS to store encrypted DLLs and configuration data alongside innocent-looking desktop.ini files. This technique bypassed file-based antivirus scanning because the ADS content was not examined during standard file scans. The campaign affected thousands of users before researchers at Microsoft's Defender ATP team identified the ADS-based hiding technique.

## What You'll Learn
- How NTFS Alternate Data Streams work and why they exist
- How attackers use ADS to hide malicious payloads
- How to detect ADS-based hiding using Python (cross-platform simulation)

In [ ]:
# EDUCATIONAL PURPOSE ONLY
# Cross-platform simulation of NTFS ADS concepts
# Real ADS only works on NTFS (Windows) — we simulate the concept on all platforms

import os
import json
import hashlib
import tempfile
import platform
from datetime import datetime
from pathlib import Path

IS_WINDOWS = platform.system() == 'Windows'
SANDBOX = tempfile.mkdtemp(prefix='cyberlab_ads_')

print(f'Platform: {platform.system()}')
print(f'NTFS ADS natively available: {IS_WINDOWS}')
print(f'Sandbox: {SANDBOX}')
print('Using simulation mode for cross-platform compatibility.')

## 1. How Alternate Data Streams Work

On NTFS, every file has a default data stream (`$DATA`). ADS allows additional named streams to be attached:

```
document.txt              <- default stream (visible)
document.txt:hidden.exe   <- alternate data stream (invisible to dir/Explorer)
document.txt:config.dat   <- another alternate data stream
```

Windows uses ADS legitimately: the Zone.Identifier stream marks files downloaded from the internet. Attackers exploit the same mechanism to hide payloads.

In [ ]:
# EDUCATIONAL PURPOSE ONLY
# Simulate ADS structure (cross-platform)

class ADSSimulator:
    """Simulates NTFS Alternate Data Streams for educational purposes.
    On Windows with NTFS, could use real ADS. Uses JSON simulation everywhere."""
    
    def __init__(self, base_dir):
        self.base_dir = base_dir
        self.ads_registry = {}  # filepath -> {stream_name: content}
    
    def create_file(self, filename, content):
        """Create a normal file (default stream)."""
        filepath = os.path.join(self.base_dir, filename)
        with open(filepath, 'w') as f:
            f.write(content)
        self.ads_registry[filepath] = {'$DATA': content}
        return filepath
    
    def write_ads(self, filepath, stream_name, content):
        """Write data to an alternate data stream."""
        if filepath not in self.ads_registry:
            self.ads_registry[filepath] = {}
        
        self.ads_registry[filepath][stream_name] = content
        
        # On real NTFS, this would be:
        # with open(f'{filepath}:{stream_name}', 'w') as f:
        #     f.write(content)
        
        # Store simulation data in a hidden metadata file
        meta_path = filepath + '.ads_sim.json'
        streams = {}
        if os.path.exists(meta_path):
            with open(meta_path, 'r') as f:
                streams = json.load(f)
        streams[stream_name] = {
            'content': content,
            'size': len(content),
            'hash': hashlib.sha256(content.encode()).hexdigest(),
            'created': datetime.now().isoformat(),
        }
        with open(meta_path, 'w') as f:
            json.dump(streams, f, indent=2)
        
        return True
    
    def read_ads(self, filepath, stream_name):
        """Read from an alternate data stream."""
        if filepath in self.ads_registry:
            return self.ads_registry[filepath].get(stream_name)
        return None
    
    def list_streams(self, filepath):
        """List all streams for a file (including ADS)."""
        if filepath in self.ads_registry:
            return list(self.ads_registry[filepath].keys())
        return ['$DATA']
    
    def get_visible_size(self, filepath):
        """Size visible to standard tools (default stream only)."""
        return os.path.getsize(filepath)
    
    def get_total_size(self, filepath):
        """Total size including all ADS content."""
        total = os.path.getsize(filepath)
        if filepath in self.ads_registry:
            for name, content in self.ads_registry[filepath].items():
                if name != '$DATA':
                    total += len(content)
        return total

ads = ADSSimulator(SANDBOX)

# Create a normal-looking file
readme_path = ads.create_file('readme.txt', 'This is a normal readme file.\n')
print(f'Created: readme.txt ({ads.get_visible_size(readme_path)} bytes)')
print(f'Streams: {ads.list_streams(readme_path)}')

## 2. Simulating the Astaroth Attack Pattern

Astaroth hid encrypted DLLs, configuration data, and command output inside ADS attached to benign files. We simulate this pattern to understand what defenders need to look for.

In [ ]:
# EDUCATIONAL PURPOSE ONLY
# Simulate Astaroth-style ADS hiding

# Create several "normal" files
files = {
    'desktop.ini': '[.ShellClassInfo]\nIconResource=shell32.dll,3\n',
    'thumbs.db': 'SIMULATED_THUMBNAIL_CACHE_DATA',
    'notes.txt': 'Meeting notes for Q3 planning session.\n',
}

created_paths = {}
for name, content in files.items():
    created_paths[name] = ads.create_file(name, content)

# Attacker hides payloads in ADS
attack_streams = [
    (created_paths['desktop.ini'], 'payload.dll',
     'TVqQAAMAAAAEAAAA//8AALgAAAA' * 100),  # Simulated PE header (not real)
    (created_paths['desktop.ini'], 'config.dat',
     json.dumps({'c2': 'example.com', 'interval': 300, 'key': 'SIMULATED_KEY'})),
    (created_paths['thumbs.db'], 'exfil.dat',
     'SIMULATED_STOLEN_CREDENTIALS_DATA' * 20),
    (created_paths['notes.txt'], 'Zone.Identifier',
     '[ZoneTransfer]\nZoneId=3\nHostUrl=http://example.com\n'),  # Legitimate ADS
]

print('=== ADS Injection Simulation ===')
print(f'{"File":<20s} {"Stream":<20s} {"Hidden Size":<15s} {"Type"}')
print('-' * 70)

for filepath, stream, content in attack_streams:
    ads.write_ads(filepath, stream, content)
    filename = os.path.basename(filepath)
    is_suspicious = stream not in ('Zone.Identifier',)
    tag = 'MALICIOUS' if is_suspicious else 'Legitimate'
    print(f'{filename:<20s} {stream:<20s} {len(content):<15d} {tag}')

# Show the size discrepancy
print('\n=== Size Analysis ===')
for name, path in created_paths.items():
    visible = ads.get_visible_size(path)
    total = ads.get_total_size(path)
    hidden = total - visible
    flag = ' <-- SUSPICIOUS' if hidden > 100 else ''
    print(f'{name:<20s} Visible: {visible:>6d}B  Total: {total:>6d}B  Hidden: {hidden:>6d}B{flag}')

## 3. ADS Detection Scanner

Standard file listings miss ADS content entirely. A dedicated scanner must enumerate all streams and flag unexpected ones.

In [ ]:
# EDUCATIONAL PURPOSE ONLY
# ADS detection scanner

# Known legitimate ADS names
LEGITIMATE_STREAMS = {'$DATA', 'Zone.Identifier', 'encryptable', 'OECustomProperty'}

def scan_for_hidden_streams(simulator):
    """Scan all tracked files for suspicious alternate data streams."""
    findings = []
    
    for filepath, streams in simulator.ads_registry.items():
        for stream_name, content in streams.items():
            if stream_name in LEGITIMATE_STREAMS:
                continue
            
            content_str = content if isinstance(content, str) else str(content)
            
            # Heuristic checks
            suspicious_indicators = []
            if len(content_str) > 1000:
                suspicious_indicators.append('large_content')
            if 'TVqQ' in content_str:  # PE header signature (base64)
                suspicious_indicators.append('possible_executable')
            if 'c2' in content_str.lower() or 'beacon' in content_str.lower():
                suspicious_indicators.append('c2_config_keywords')
            if stream_name.endswith(('.dll', '.exe', '.dat', '.bin')):
                suspicious_indicators.append('suspicious_extension')
            
            if suspicious_indicators:
                severity = 'CRITICAL' if 'possible_executable' in suspicious_indicators else 'HIGH'
            else:
                severity = 'MEDIUM'
            
            findings.append({
                'file': os.path.basename(filepath),
                'stream': stream_name,
                'size': len(content_str),
                'severity': severity,
                'indicators': suspicious_indicators,
                'hash': hashlib.sha256(content_str.encode()).hexdigest()[:16],
            })
    
    return findings

scan_results = scan_for_hidden_streams(ads)

print(f'=== ADS Scanner Results ===')
print(f'Files scanned: {len(ads.ads_registry)}')
print(f'Suspicious streams found: {len(scan_results)}\n')

for finding in scan_results:
    print(f"[{finding['severity']:8s}] {finding['file']}:{finding['stream']}")
    print(f"           Size: {finding['size']}B | Hash: {finding['hash']}...")
    if finding['indicators']:
        print(f"           Indicators: {', '.join(finding['indicators'])}")
    print()

In [ ]:
# EDUCATIONAL PURPOSE ONLY
# Visualization: ADS hiding analysis

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0F0F0F')
fig.suptitle('Alternate Data Stream Analysis', color='#EDEAE4', fontsize=14, y=1.02)

# Left: Visible vs hidden data per file
ax1.set_facecolor('#1A1A1A')
file_names = list(created_paths.keys())
visible_sizes = [ads.get_visible_size(created_paths[n]) for n in file_names]
hidden_sizes = [ads.get_total_size(created_paths[n]) - ads.get_visible_size(created_paths[n])
                for n in file_names]

x = range(len(file_names))
width = 0.35
ax1.bar([i - width/2 for i in x], visible_sizes, width,
        label='Visible (default stream)', color='#6C9BCF', edgecolor='#282828')
ax1.bar([i + width/2 for i in x], hidden_sizes, width,
        label='Hidden (ADS)', color='#CF6C6C', edgecolor='#282828')
ax1.set_xticks(list(x))
ax1.set_xticklabels(file_names, fontsize=9)
ax1.set_ylabel('Size (bytes)', color='#7A7570')
ax1.set_title('Visible vs Hidden Data', color='#EDEAE4', fontsize=12)
ax1.legend(facecolor='#1A1A1A', edgecolor='#282828', labelcolor='#EDEAE4', fontsize=9)
ax1.tick_params(colors='#7A7570')
for spine in ax1.spines.values():
    spine.set_edgecolor('#282828')

# Right: Detection severity distribution
ax2.set_facecolor('#1A1A1A')
sev_counts = {}
for f in scan_results:
    sev_counts[f['severity']] = sev_counts.get(f['severity'], 0) + 1
sev_colors = {'CRITICAL': '#CF6C6C', 'HIGH': '#CF9B6C', 'MEDIUM': '#6C9BCF'}
labels = list(sev_counts.keys())
sizes = list(sev_counts.values())
pie_colors = [sev_colors.get(l, '#7A7570') for l in labels]
if sizes:
    ax2.pie(sizes, labels=labels, colors=pie_colors, autopct='%1.0f%%',
            textprops={'color': '#EDEAE4', 'fontsize': 11},
            wedgeprops={'edgecolor': '#282828'})
ax2.set_title('Detection Severity', color='#EDEAE4', fontsize=12)

plt.tight_layout()
plt.show()

## Defender's Perspective

| Indicator | Type | Detection |
|---|---|---|
| Files with ADS larger than the default stream | Anomaly | `dir /r` on Windows or PowerShell `Get-Item -Stream *` |
| ADS with executable content (PE headers, scripts) | Content | Stream-aware AV scanning, YARA rules on ADS content |
| ADS creation events in Sysmon logs | Event | Sysmon Event ID 15 (FileCreateStreamHash) |
| Zone.Identifier ADS missing from downloaded files | Absence | Indicates ADS stripping to bypass Mark-of-the-Web |

## Article Seed

**Title:** The Invisible Filesystem: How Astaroth Hid Malware in Plain Sight Using NTFS Streams

**Opening:** Your antivirus scans every file on disk — but what if the malware is stored in a part of the file that most tools don't even look at? NTFS Alternate Data Streams have been a favorite hiding spot for malware authors since the early 2000s, and the Astaroth trojan showed in 2019 that the technique is as effective as ever.

**Sections:**
1. NTFS Alternate Data Streams: the feature Microsoft built for compatibility
2. How Astaroth weaponized ADS to evade detection
3. Building an ADS scanner in Python
4. Sysmon, PowerShell, and YARA: three tools for ADS detection

**Tags:** cybersecurity, evasion, ntfs, malware-analysis, windows-security

In [ ]:
# EDUCATIONAL PURPOSE ONLY
# Self-check assertions

import shutil

# Verify ADS simulator created correct structure
assert len(ads.ads_registry) >= 4, 'Should have at least 4 tracked files'

# Verify hidden streams were created
desktop_streams = ads.list_streams(created_paths['desktop.ini'])
assert 'payload.dll' in desktop_streams, 'Should have payload.dll ADS'
assert 'config.dat' in desktop_streams, 'Should have config.dat ADS'

# Verify scanner detected suspicious streams
assert len(scan_results) >= 3, f'Expected at least 3 findings, got {len(scan_results)}'

# Verify severity classification
critical = [f for f in scan_results if f['severity'] == 'CRITICAL']
assert len(critical) >= 1, 'Should have at least 1 CRITICAL finding (executable content)'

# Verify size discrepancy detection works
for name, path in created_paths.items():
    assert ads.get_total_size(path) >= ads.get_visible_size(path)

# Cleanup
shutil.rmtree(SANDBOX, ignore_errors=True)
print('All assertions passed.')
print('Sandbox cleaned up.')